In [1]:
import pandas as pd

# Load the parquet file
file_path = '/content/drive/MyDrive/Colab Notebooks/Colab Notebooks/Datasets/filtered_trawlers.parquet'
df = pd.read_parquet(file_path)

In [2]:
# 1. Filter out voyages with fewer than 20 points
voyage_sizes = df.groupby('voyage_id').size()
valid_voyages = voyage_sizes[voyage_sizes >= 20].index
df = df[df['voyage_id'].isin(valid_voyages)].copy()

# 2. Save this intermediate dataset as df_to_pred
df_to_pred = df.copy()

# 3. Drop all -1 points ('is_fishing' == -1 indicates unknown/unlabeled points)
df = df[df['is_fishing'] != -1].copy()

print(f"Initial dataset size (df_to_pred): {len(df_to_pred):,}")
print(f"Training dataset size after removing -1 points: {len(df):,}")

Initial dataset size (df_to_pred): 2,993,216
Training dataset size after removing -1 points: 117,287


In [3]:
# Verify if the DataFrame is sorted by 'mmsi' and 'timestamp'
is_sorted = df.equals(df.sort_values(by=['mmsi', 'timestamp']))
print(f"Is the dataset currently sorted by 'mmsi' and 'timestamp'? {is_sorted}")

Is the dataset currently sorted by 'mmsi' and 'timestamp'? True


## CELL 1 — Imports and reproducibility
This cell imports all required libraries, configures the random seeds for reproducibility across Python, NumPy, and TensorFlow, and imports the necessary Keras and scikit-learn components.

In [4]:
import random
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dropout, Dense, LayerNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

## CELL 2 — Convert target to binary
We convert the continuous target variable `is_fishing` to a binary target variable `is_fishing_binary` using a threshold of 0.5.

In [5]:
# Convert continuous column to binary classification target
df['is_fishing_binary'] = (df['is_fishing'] >= 0.5).astype(int)

min_fishing = df['is_fishing'].min()
max_fishing = df['is_fishing'].max()
class_counts = df['is_fishing_binary'].value_counts().sort_index()
class_percentages = df['is_fishing_binary'].value_counts(normalize=True).sort_index() * 100

print(f"Original 'is_fishing' minimum: {min_fishing}")
print(f"Original 'is_fishing' maximum: {max_fishing}\n")
for c in [0, 1]:
    label = "Non-Fishing (0)" if c == 0 else "Fishing (1)"
    print(f"{label}: {class_counts[c]:,} samples ({class_percentages[c]:.2f}%)")

Original 'is_fishing' minimum: 0.0
Original 'is_fishing' maximum: 1.0

Non-Fishing (0): 60,920 samples (51.94%)
Fishing (1): 56,367 samples (48.06%)


## CELL 3 — Select input features
We verify and choose the 13 defined model input features, ensuring no missing values exist.

In [6]:
feature_cols = [
    "speed",
    "course",
    "dt",
    "log_dt",
    "distance_from_shore",
    "distance_from_port",
    "log_distance_from_shore",
    "log_distance_from_port",
    "d_course",
    "dx",
    "dy",
    "vx",
    "vy"
]

# Verify features exist
for col in feature_cols:
    if col not in df.columns:
        raise ValueError(f"Feature {col} is missing from the DataFrame.")

# Check for missing values
missing_vals = df[feature_cols + ['is_fishing_binary']].isnull().sum()
print("Missing values check:")
print(missing_vals)

print(f"\nFinal number of selected input features: {len(feature_cols)}")

Missing values check:
speed                      0
course                     0
dt                         0
log_dt                     0
distance_from_shore        0
distance_from_port         0
log_distance_from_shore    0
log_distance_from_port     0
d_course                   0
dx                         0
dy                         0
vx                         0
vy                         0
is_fishing_binary          0
dtype: int64

Final number of selected input features: 13


## CELL 4 — Split by voyage_id
To prevent data leakage, we split our data at the voyage level. We shuffle the unique voyage IDs using a fixed seed, assigning 70% to training, 15% to validation, and 15% to test.

In [7]:
SEED = 42
rng = np.random.default_rng(SEED)
unique_voyages = df['voyage_id'].unique()
rng.shuffle(unique_voyages)

n_voyages = len(unique_voyages)
train_split = int(0.70 * n_voyages)
val_split = int(0.85 * n_voyages)

train_voyages = unique_voyages[:train_split]
val_voyages = unique_voyages[train_split:val_split]
test_voyages = unique_voyages[val_split:]

df_train = df[df['voyage_id'].isin(train_voyages)].copy()
df_val = df[df['voyage_id'].isin(val_voyages)].copy()
df_test = df[df['voyage_id'].isin(test_voyages)].copy()

print(f"Train voyages: {len(train_voyages)}")
print(f"Validation voyages: {len(val_voyages)}")
print(f"Test voyages: {len(test_voyages)}\n")

print(f"Train rows: {len(df_train):,}")
print(f"Validation rows: {len(df_val):,}")
print(f"Test rows: {len(df_test):,}\n")

print(f"Train MMSIs: {df_train['mmsi'].nunique()}")
print(f"Validation MMSIs: {df_val['mmsi'].nunique()}")
print(f"Test MMSIs: {df_test['mmsi'].nunique()}\n")

print("Train Class Distribution:")
print(df_train['is_fishing_binary'].value_counts(normalize=True))
print("\nValidation Class Distribution:")
print(df_val['is_fishing_binary'].value_counts(normalize=True))
print("\nTest Class Distribution:")
print(df_test['is_fishing_binary'].value_counts(normalize=True))

NameError: name 'SEED' is not defined

## CELL 5 — Scale the features
We fit the `StandardScaler` strictly on the training partition and transform the train, validation, and test datasets in consequence.

In [ ]:
scaler = StandardScaler()

# Fit on training data and transform
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# Transform validation and test data
df_val[feature_cols] = scaler.transform(df_val[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

print("Features scaled successfully.")

## CELL 6 — Create many-to-many sequences
We build sequence slices of 10 timesteps with stride 1, sorted chronologically and grouped by voyage to ensure sequences never cross voyage boundaries.

In [ ]:
def create_sequences(dataframe, feature_cols, sequence_length=10, stride=1):
    X_seq, y_seq = [], []
    # Group by voyage_id
    grouped = dataframe.groupby('voyage_id')
    for _, group in grouped:
        # Sort chronologically
        group_sorted = group.sort_values('timestamp')
        features = group_sorted[feature_cols].values
        targets = group_sorted['is_fishing_binary'].values

        # Create rolling window sequences
        n_samples = len(group_sorted)
        for i in range(0, n_samples - sequence_length + 1, stride):
            X_seq.append(features[i : i + sequence_length])
            y_seq.append(targets[i : i + sequence_length])

    return np.array(X_seq), np.array(y_seq)[..., np.newaxis]

sequence_length = 10
stride = 1

X_train, y_train = create_sequences(df_train, feature_cols, sequence_length, stride)
X_val, y_val = create_sequences(df_val, feature_cols, sequence_length, stride)
X_test, y_test = create_sequences(df_test, feature_cols, sequence_length, stride)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape} | y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}\n")

print(f"Train sequences: {len(X_train):,}")
print(f"Validation sequences: {len(X_val):,}")
print(f"Test sequences: {len(X_test):,}")

## CELL 7 — Build the LSTM model
We instantiate our many-to-many Bidirectional LSTM architecture.

In [ ]:
binary_model = Sequential([
    Input(shape=(sequence_length, len(feature_cols))),

    Bidirectional(LSTM(64, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.2),

    Bidirectional(LSTM(32, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.2),

    Dense(16, activation="relu"),
    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

binary_model.summary()

## CELL 8 — Compile and train
We set up compiling configurations with early stopping and model checkpointing to preserve the best validation model.

In [ ]:
binary_model.compile(
    optimizer=Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_binary_lstm_model.keras', save_best_only=True, monitor='val_loss', verbose=1)
]

history = binary_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

## CELL 9 — Generate test predictions
We use the loaded best checkpoint to evaluate probabilities on our held-out test data.

In [ ]:
# Load the best saved model weights
binary_model.load_weights('best_binary_lstm_model.keras')

# Predict probabilities
y_pred_prob = binary_model.predict(X_test)

# Convert probabilities to classes
y_pred_class = (y_pred_prob >= 0.5).astype(int)

# Flatten to calculate evaluation metrics
y_test_flat = y_test.flatten()
y_pred_prob_flat = y_pred_prob.flatten()
y_pred_class_flat = y_pred_class.flatten()

print(f"Flat True Test Targets Shape: {y_test_flat.shape}")
print(f"Flat Pred Test Targets Shape: {y_pred_class_flat.shape}")
assert len(y_test_flat) == len(y_pred_class_flat), "Prediction counts must match target counts!"

## CELL 10 — Classification evaluation
We evaluate performance metrics using scikit-learn metrics.

In [ ]:
acc = accuracy_score(y_test_flat, y_pred_class_flat)
prec = precision_score(y_test_flat, y_pred_class_flat)
rec = recall_score(y_test_flat, y_pred_class_flat)
f1 = f1_score(y_test_flat, y_pred_class_flat)
auc = roc_auc_score(y_test_flat, y_pred_prob_flat)
cm = confusion_matrix(y_test_flat, y_pred_class_flat)

print("Classification Report:")
print(classification_report(y_test_flat, y_pred_class_flat))

print(f"Total Test Timestep Predictions: {len(y_test_flat):,}")
print(f"Class 0 Support (Non-Fishing): {np.sum(y_test_flat == 0):,}")
print(f"Class 1 Support (Fishing): {np.sum(y_test_flat == 1):,}")

## CELL 11 — Diagnostic visualizations
We generate four diagnostic plots along with sequence inspections to evaluate how predictions track with true transitions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Loss curves
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test_flat, y_pred_prob_flat)
axes[0, 1].plot(fpr, tpr, label=f'ROC AUC = {auc:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

# Plot 3: Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test_flat, y_pred_prob_flat)
axes[1, 0].plot(recall_vals, precision_vals, label='PR Curve')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

# Plot 4: Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Non-Fishing (0)', 'Fishing (1)'],
            yticklabels=['Non-Fishing (0)', 'Fishing (1)'])
axes[1, 1].set_title('Confusion Matrix')
axes[1, 1].set_ylabel('True Label')
axes[1, 1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Sequence visual inspections
print("\n--- Example Sequence Visual Inspections (10-ping Sequences) ---")
sample_indices = [0, len(X_test)//2, len(X_test)-1]
for idx in sample_indices:
    print(f"\nSequence Sample Index: {idx}")
    print("Timestep | True Label | Predicted Probability | Predicted Class")
    for t in range(sequence_length):
        true_lbl = int(y_test[idx, t, 0])
        pred_prob = float(y_pred_prob[idx, t, 0])
        pred_cls = int(y_pred_class[idx, t, 0])
        print(f"{t:8d} | {true_lbl:10d} | {pred_prob:21.4f} | {pred_cls:15d}")

## CELL 12 — Save artifacts
We serialize the necessary components to guarantee model inference portability.

In [ ]:
# Save objects
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('binary_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

config = {
    'SEED': SEED,
    'sequence_length': sequence_length,
    'stride': stride,
    'target_threshold': 0.5,
    'feature_cols': feature_cols
}
with open('experiment_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("All artifacts and config successfully saved.")

## CELL 13 — Final experiment summary
This prints the required experiment summary.

In [ ]:
summary_report = f"""
========================================
AIS FISHING CLASSIFICATION LSTM
========================================

Target:
is_fishing < 0.5  → 0
is_fishing >= 0.5 → 1

Split:
70% voyages → Train ({len(train_voyages)} voyages)
15% voyages → Validation ({len(val_voyages)} voyages)
15% voyages → Test ({len(test_voyages)} voyages)

Sequence length: {sequence_length}
Stride: {stride}
Features: {len(feature_cols)}

Model:
Bidirectional LSTM 64
Bidirectional LSTM 32
Dense 16
Sigmoid output

Test Results:
Accuracy:  {acc:.4f}
Precision: {prec:.4f}
Recall:    {rec:.4f}
F1:        {f1:.4f}
ROC-AUC:   {auc:.4f}

Confusion Matrix:
{cm}

Class Distribution:
Class 0: {np.sum(y_test_flat == 0):,}
Class 1: {np.sum(y_test_flat == 1):,}
"""
print(summary_report)

In [ ]:
import numpy as np
import pandas as pd

# Set reproducibility seeds (re-defining SEED in case it was missing)
SEED = 42

# Prepare the entire dataset for predictions using the fitted scaler
df_scaled = df_to_pred.copy()
df_scaled[feature_cols] = scaler.transform(df_to_pred[feature_cols])

print("Generating predictions for the entire dataset...")
predictions_list = []
voyage_groups = df_scaled.groupby('voyage_id', sort=False)

for voyage_id, group in voyage_groups:
    # Ensure chronological order within voyage
    group_sorted = group.sort_values('timestamp')
    features = group_sorted[feature_cols].values

    n_samples = len(group_sorted)
    if n_samples == 0:
        continue

    # Pad features at the start with the first row repeated (sequence_length - 1) times
    padding = np.repeat(features[0:1], sequence_length - 1, axis=0)
    padded_features = np.vstack([padding, features])

    # Create sliding windows
    voyage_sequences = []
    for i in range(n_samples):
        voyage_sequences.append(padded_features[i : i + sequence_length])

    voyage_sequences = np.array(voyage_sequences)

    # Predict for this voyage
    pred_probs = binary_model.predict(voyage_sequences, verbose=0)
    # Extract the prediction corresponding to the last timestep of each sequence window
    last_timestep_preds = pred_probs[:, -1, 0]
    # Convert probabilities to binary labels
    binary_preds = (last_timestep_preds >= 0.5).astype(int)

    # Add back to list in original chronological order of this voyage
    predictions_list.extend(binary_preds)

# Store predictions into the new column 'LSTM_Ping_labels'
df_to_pred['LSTM_Ping_labels'] = predictions_list
print("Predictions generated successfully!")
display(df_to_pred[['mmsi', 'timestamp', 'voyage_id', 'LSTM_Ping_labels']].head())

Generating predictions for the entire dataset...
Predictions generated successfully!


,mmsi,timestamp,voyage_id,LSTM_Ping_labels
43,1.252340e+12,1.328393e+09,1252339803566_17,0
44,1.252340e+12,1.328395e+09,1252339803566_17,0
45,1.252340e+12,1.328396e+09,1252339803566_17,0
46,1.252340e+12,1.328398e+09,1252339803566_17,0
47,1.252340e+12,1.328402e+09,1252339803566_17,0


In [ ]:
import os

# Define output paths
output_dir = '/content/drive/MyDrive/Colab Notebooks/Colab Notebooks/Datasets/'
os.makedirs(output_dir, exist_ok=True)

# 1. Save dataset (filtered) without labels as 'Seq_filtered_trawlers.parquet'
label_cols_to_remove = [col for col in ['is_fishing', 'is_fishing_binary', 'LSTM_Ping_labels'] if col in df_to_pred.columns]
df_without_labels = df_to_pred.drop(columns=label_cols_to_remove)
df_without_labels.to_parquet(os.path.join(output_dir, 'Seq_filtered_trawlers.parquet'), index=False)

# 2. Save the labels as a 1D column/series matching the dataset row alignment
labels_df = df_to_pred[['LSTM_Ping_labels']].copy()
labels_df.to_parquet(os.path.join(output_dir, 'LSTM_Ping_labels.parquet'), index=False)

print("Saved files successfully to:")
print(f"- {os.path.join(output_dir, 'Seq_filtered_trawlers.parquet')}")
print(f"- {os.path.join(output_dir, 'LSTM_Ping_labels.parquet')}")

Saved files successfully!

--- Verification of Saved Parquet Columns ---
Seq_filtered_trawlers.parquet (2,993,216 rows):
['mmsi', 'timestamp', 'distance_from_shore', 'distance_from_port', 'speed', 'course', 'lat', 'lon', 'is_fishing', 'source', 'dt', 'log_dt', 'log_distance_from_shore', 'log_distance_from_port', 'd_course', 'dx', 'dy', 'vx', 'vy', 'voyage_id']

LSTM_Ping_labels.parquet (2,993,216 rows):
['LSTM_Ping_labels']
